In [27]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging 
import time 

logging.basicConfig(
    level = logging.INFO,
    format= '%(asctime)s | %(name)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('scraping_uhuy.log', encoding='utf-8'), # ke file
        logging.StreamHandler()                                     # ke layar notebook
    ]
)

logger = logging.getLogger('book_scraper')
base_url = 'https://books.toscrape.com'
rating_map = {'One':1, 'Two':2, 'Three':3, 'Four':4, 'Five':5}

def fetch_page(page):
    if page == 1 :
        url = base_url
    else :
        url = f'{base_url}/catalogue/page-{page}.html'
    
    #pakai try except supaya tidak berhenti jika ada satu halaman yg gagal
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'
        return response 
    except requests.RequestException as e :
        logger.error(f'Gagal ambil halaman {page} : {e}') #bedanya kalo pakai logging maka akan muncul 'Root' bukan nama loggernya
        return None
     
def parse_book(response):
    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.find_all('article', class_='product_pod')
    result = []
    for book in books :
        title = book.h3.a.get('title')
        rating = book.find('p', class_='star-rating')['class'][1]
        harga = book.find('p', class_='price_color').get_text(strip=True)
        stok = book.find('p', class_='instock availability').get_text(strip=True)
        img_relatif = book.find('img')['src']
        img_url = urljoin(response.url,img_relatif)
        result.append({
            'Title' : title,
            'Rating' : rating,
            'Price' : harga,
            'Stok' : stok,
            'Link Image' : img_url
        })
    return result

def scrape_all(total_page, delay):
    summary = []
    for page in range(1, total_page + 1):
        response = fetch_page(page)
        if response is None:
            continue 
        summary.extend(parse_book(response))
        logger.info(f'Selesai halaman {page}. Total {len(summary)} buku')
        time.sleep(delay)
    logger.info(f'Semua buku telah di-scraping. Total {len(summary)} buku')
    return summary

def cleaning_data(df):
    df_clean = df.copy()
    df_clean['Price'] = df_clean['Price'].str.replace('£','',regex=False).astype(float)
    df_clean['Rating'] = df_clean['Rating'].map(rating_map)
    return df_clean

def run_pipeline(total_page=50, delay=0.5):
    summary = scrape_all(total_page,delay)
    df = pd.DataFrame(summary)
    df_clean = cleaning_data(df)
    return df_clean

df = run_pipeline()

2026-10-22 05:59:25,477 | book_scraper | INFO | Selesai halaman 1. Total 20 buku
2026-10-22 05:59:27,542 | book_scraper | INFO | Selesai halaman 2. Total 40 buku
2026-10-22 05:59:29,703 | book_scraper | INFO | Selesai halaman 3. Total 60 buku
2026-10-22 05:59:31,992 | book_scraper | INFO | Selesai halaman 4. Total 80 buku
2026-10-22 05:59:34,101 | book_scraper | INFO | Selesai halaman 5. Total 100 buku
2026-10-22 05:59:36,242 | book_scraper | INFO | Selesai halaman 6. Total 120 buku
2026-10-22 05:59:38,541 | book_scraper | INFO | Selesai halaman 7. Total 140 buku
2026-10-22 05:59:40,642 | book_scraper | INFO | Selesai halaman 8. Total 160 buku
2026-10-22 05:59:42,906 | book_scraper | INFO | Selesai halaman 9. Total 180 buku
2026-10-22 05:59:45,034 | book_scraper | INFO | Selesai halaman 10. Total 200 buku
2026-10-22 05:59:47,612 | book_scraper | INFO | Selesai halaman 11. Total 220 buku
2026-10-22 05:59:50,006 | book_scraper | INFO | Selesai halaman 12. Total 240 buku
2026-10-22 05:59:

In [28]:
#testing fetch_page
response = fetch_page(1)
response

<Response [200]>

In [29]:
#testing parse_book
response = fetch_page(1)
data = parse_book(response)
print(len(data))

20


In [30]:
#testing scrape_all 1 page
summary= scrape_all(1,0.5)
summary

2026-10-22 06:11:34,989 | book_scraper | INFO | Selesai halaman 1. Total 20 buku
2026-10-22 06:11:35,491 | book_scraper | INFO | Semua buku telah di-scraping. Total 20 buku


[{'Title': 'A Light in the Attic',
  'Rating': 'Three',
  'Price': '£51.77',
  'Stok': 'In stock',
  'Link Image': 'https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg'},
 {'Title': 'Tipping the Velvet',
  'Rating': 'One',
  'Price': '£53.74',
  'Stok': 'In stock',
  'Link Image': 'https://books.toscrape.com/media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg'},
 {'Title': 'Soumission',
  'Rating': 'One',
  'Price': '£50.10',
  'Stok': 'In stock',
  'Link Image': 'https://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg'},
 {'Title': 'Sharp Objects',
  'Rating': 'Four',
  'Price': '£47.82',
  'Stok': 'In stock',
  'Link Image': 'https://books.toscrape.com/media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg'},
 {'Title': 'Sapiens: A Brief History of Humankind',
  'Rating': 'Five',
  'Price': '£54.23',
  'Stok': 'In stock',
  'Link Image': 'https://books.toscrape.com/media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg'},
 {'

In [37]:
#testing running run_pipeline 1 page
df = run_pipeline(1)
df

2026-10-22 06:14:00,956 | book_scraper | INFO | Selesai halaman 1. Total 20 buku
2026-10-22 06:14:01,458 | book_scraper | INFO | Semua buku telah di-scraping. Total 20 buku


,Title,Rating,Price,Stok,Link Image
0,A Light in the Attic,3,51.77,In stock,https://books.toscrape.com/media/cache/2c/da/2...
1,Tipping the Velvet,1,53.74,In stock,https://books.toscrape.com/media/cache/26/0c/2...
2,Soumission,1,50.10,In stock,https://books.toscrape.com/media/cache/3e/ef/3...
3,Sharp Objects,4,47.82,In stock,https://books.toscrape.com/media/cache/32/51/3...
4,Sapiens: A Brief History of Humankind,5,54.23,In stock,https://books.toscrape.com/media/cache/be/a5/b...
5,The Requiem Red,1,22.65,In stock,https://books.toscrape.com/media/cache/68/33/6...
6,The Dirty Little Secrets of Getting Your Dream...,4,33.34,In stock,https://books.toscrape.com/media/cache/92/27/9...
7,The Coming Woman: A Novel Based on the Life of...,3,17.93,In stock,https://books.toscrape.com/media/cache/3d/54/3...
8,The Boys in the Boat: Nine Americans and Their...,4,22.60,In stock,https://books.toscrape.com/media/cache/66/88/6...
9,The Black Maria,1,52.15,In stock,https://books.toscrape.com/media/cache/58/46/5...


In [34]:
# Cek Sampling Link Image
df[df['Title'] == 'Soumission']['Link Image'].iloc()[0] #pakai .iloc()[0] agar hasilnya tidak terpotong


'https://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg'

In [35]:
df

,Title,Rating,Price,Stok,Link Image
0,A Light in the Attic,3,51.77,In stock,https://books.toscrape.com/media/cache/2c/da/2...
1,Tipping the Velvet,1,53.74,In stock,https://books.toscrape.com/media/cache/26/0c/2...
2,Soumission,1,50.10,In stock,https://books.toscrape.com/media/cache/3e/ef/3...
3,Sharp Objects,4,47.82,In stock,https://books.toscrape.com/media/cache/32/51/3...
4,Sapiens: A Brief History of Humankind,5,54.23,In stock,https://books.toscrape.com/media/cache/be/a5/b...
5,The Requiem Red,1,22.65,In stock,https://books.toscrape.com/media/cache/68/33/6...
6,The Dirty Little Secrets of Getting Your Dream...,4,33.34,In stock,https://books.toscrape.com/media/cache/92/27/9...
7,The Coming Woman: A Novel Based on the Life of...,3,17.93,In stock,https://books.toscrape.com/media/cache/3d/54/3...
8,The Boys in the Boat: Nine Americans and Their...,4,22.60,In stock,https://books.toscrape.com/media/cache/66/88/6...
9,The Black Maria,1,52.15,In stock,https://books.toscrape.com/media/cache/58/46/5...
